# 03. Feature Engineering

## Objective

This notebook prepares the machine learning dataset for model training by performing feature engineering and preprocessing.

### Goals

- Load cleaned dataset from previous phase
- Handle data type corrections
- Create derived features
- Encode categorical variables
- Scale numerical variables
- Perform feature selection
- Save processed dataset for model training

This notebook produces the final ML-ready dataset.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# ============================================================
# Standard Library
# ============================================================

import warnings
warnings.filterwarnings("ignore")

# ============================================================
# Third Party Libraries
# ============================================================

import numpy as np
import pandas as pd
import joblib

# ============================================================
# Project Modules
# ============================================================

from src.config.paths import (
    INTERIM_DATA_DIR,
    PREPROCESSOR_PATH,
    PROCESSED_DATASET_PATH
)

from src.config.settings import (
    RANDOM_STATE,
    TARGET_COLUMN
)

from src.data.data_loader import save_dataset

from src.features.feature_creator import create_all_features
from src.features.preprocessor_builder import (
    identify_feature_types,
    apply_preprocessor
)

# ============================================================
# Display Configuration
# ============================================================

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

def print_section(title):
    print("=" * 60)
    print(title)
    print("=" * 60)

print("Libraries Imported Successfully.")

Libraries Imported Successfully.


In [2]:
# ============================================================
# Load Interim Dataset
# ============================================================

df = pd.read_csv(INTERIM_DATA_DIR / "ml_dataset_interim.csv")

print_section("LOADED INTERIM DATASET")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
display(df.head())

LOADED INTERIM DATASET
Shape: (7021, 21)
Columns: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
# ============================================================
# Data Type Corrections
# ============================================================

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())

df["SeniorCitizen"] = df["SeniorCitizen"].astype(int)

# Drop customerID if present (not a model feature)
if "customerID" in df.columns:
    df = df.drop(columns=["customerID"])

# Encode binary target
if df[TARGET_COLUMN].dtype == object:
    df[TARGET_COLUMN] = df[TARGET_COLUMN].map({"Yes": 1, "No": 0}).astype(int)

print("Data type corrections applied.")
display(df.dtypes)

Data type corrections applied.


gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
Churn                 int64
dtype: object

## Derived Feature Engineering

All derived features are created by `src.features.feature_creator.create_all_features()`, which applies:

| Feature | Logic |
|---------|-------|
| `TenureGroup` | Fixed lifecycle bins (New → Long-Term) |
| `MonthlyChargeGroup` | Quartile-based spending tiers |
| `TotalServices` | Count of active service subscriptions |
| `AvgMonthlySpend` | TotalCharges / tenure |
| `HighValueCustomer` | Above-median MonthlyCharges flag |
| `SecurityRisk` | No OnlineSecurity + No TechSupport |
| `StreamingUser` | StreamingTV or StreamingMovies active |

In [4]:
# ============================================================
# Create All Derived Features
# ============================================================

df_fe = create_all_features(df)

engineered_columns = [
    "TenureGroup", "MonthlyChargeGroup", "TotalServices",
    "AvgMonthlySpend", "HighValueCustomer", "SecurityRisk", "StreamingUser"
]

print_section("ENGINEERED FEATURES")
print(f"Total Features After Engineering: {df_fe.shape[1]}")
display(df_fe[engineered_columns].head(10))

ENGINEERED FEATURES
Total Features After Engineering: 27


,TenureGroup,MonthlyChargeGroup,TotalServices,AvgMonthlySpend,HighValueCustomer,SecurityRisk,StreamingUser
0,New,Low,2,29.850000,0,1,0
1,Established,Medium,4,55.573529,0,0,0
2,New,Medium,4,54.075000,0,0,0
3,Established,Medium,4,40.905556,0,0,0
4,New,High,2,75.825000,1,1,0
5,New,Premium,6,102.562500,1,1,1
6,Emerging,High,5,88.609091,1,1,1
7,New,Low,2,30.190000,0,0,0
8,Established,Premium,7,108.787500,1,0,1
9,Long-Term,Medium,4,56.257258,0,0,0


In [5]:
# ============================================================
# Individual Feature Distribution Previews
# ============================================================

print("Tenure Groups:")
display(df_fe["TenureGroup"].value_counts().sort_index())

print("\nMonthly Charge Groups:")
display(df_fe["MonthlyChargeGroup"].value_counts())

print("\nTotal Services Distribution:")
display(df_fe["TotalServices"].describe())

print("\nHigh Value Customer Distribution:")
display(df_fe["HighValueCustomer"].value_counts())

print("\nSecurity Risk Distribution:")
display(df_fe["SecurityRisk"].value_counts())

print("\nStreaming User Distribution:")
display(df_fe["StreamingUser"].value_counts())

Tenure Groups:


TenureGroup
New            2164
Emerging       1024
Established    1594
Loyal           832
Long-Term      1407
Name: count, dtype: int64


Monthly Charge Groups:


MonthlyChargeGroup
Low        1761
High       1756
Medium     1754
Premium    1750
Name: count, dtype: int64


Total Services Distribution:


...

HighValueCustomer
1    3515
0    3506
Name: count, dtype: int64


Security Risk Distribution:


SecurityRisk
0    4476
1    2545
Name: count, dtype: int64


Streaming User Distribution:


StreamingUser
0    3522
1    3499
Name: count, dtype: int64

## Preprocessing Pipeline

The preprocessing pipeline is constructed by `src.features.preprocessor_builder`:

- **Numerical features** → `StandardScaler`
- **Categorical features** → `OneHotEncoder` (handle_unknown='ignore')
- Binned columns (`TenureGroup`, `MonthlyChargeGroup`) are dropped to avoid collinearity.

In [6]:
# ============================================================
# Identify Feature Types (before preprocessing)
# ============================================================

# Temporarily separate to inspect types
X_temp = df_fe.drop(columns=[TARGET_COLUMN, "TenureGroup", "MonthlyChargeGroup"])
numerical_features, categorical_features = identify_feature_types(X_temp)

print_section("FEATURE TYPES")
print(f"Numerical ({len(numerical_features)}): {numerical_features}")
print(f"Categorical ({len(categorical_features)}): {categorical_features}")
print(f"Total: {len(numerical_features) + len(categorical_features)}")

FEATURE TYPES
Numerical (9): ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'TotalServices', 'AvgMonthlySpend', 'HighValueCustomer', 'SecurityRisk', 'StreamingUser']
Categorical (15): ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
Total: 24


In [7]:
# ============================================================
# Missing Value Check
# ============================================================

missing = X_temp.isnull().sum()
missing = missing[missing > 0]

if len(missing) == 0:
    print("No Missing Values Found.")
else:
    display(missing)

No Missing Values Found.


In [8]:
# ============================================================
# Build & Apply Preprocessing Pipeline
# ============================================================

df_processed, preprocessor = apply_preprocessor(
    df_fe,
    target_column=TARGET_COLUMN,
    drop_columns=["TenureGroup", "MonthlyChargeGroup"]
)

print_section("PROCESSED DATASET")
print(f"Shape: {df_processed.shape}")
print(f"Features: {df_processed.shape[1] - 1} (excluding target)")
display(df_processed.head())

PROCESSED DATASET
Shape: (7021, 51)
Features: 50 (excluding target)


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,TotalServices,AvgMonthlySpend,HighValueCustomer,SecurityRisk,StreamingUser,gender_Female,gender_Male,Partner_No,Partner_Yes,Dependents_No,Dependents_Yes,PhoneService_No,PhoneService_Yes,MultipleLines_No,MultipleLines_No phone service,MultipleLines_Yes,InternetService_DSL,InternetService_Fiber optic,InternetService_No,OnlineSecurity_No,OnlineSecurity_No internet service,OnlineSecurity_Yes,OnlineBackup_No,OnlineBackup_No internet service,OnlineBackup_Yes,DeviceProtection_No,DeviceProtection_No internet service,DeviceProtection_Yes,TechSupport_No,TechSupport_No internet service,TechSupport_Yes,StreamingTV_No,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_Month-to-month,Contract_One year,Contract_Two year,PaperlessBilling_No,PaperlessBilling_Yes,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,Churn
0,-0.440508,-1.282728,-1.164135,-0.995686,-0.932578,-1.160312,-1.001283,1.326176,-0.996729,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0
1,-0.440508,0.062387,-0.262811,-0.175262,-0.067062,-0.307607,-1.001283,-0.754048,-0.996729,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0
2,-0.440508,-1.241967,-0.365914,-0.961142,-0.067062,-0.357282,-1.001283,-0.754048,-0.996729,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1
3,-0.440508,0.510759,-0.750058,-0.196769,-0.067062,-0.793833,-1.001283,-0.754048,-0.996729,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0
4,-0.440508,-1.241967,0.194503,-0.941951,-0.932578,0.363705,0.998719,1.326176,-0.996729,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1


In [9]:
# ============================================================
# Final Dataset Validation
# ============================================================

print_section("FINAL DATASET VALIDATION")
print(f"Dataset Shape      : {df_processed.shape}")
print(f"Duplicate Rows     : {df_processed.duplicated().sum()}")
print(f"Missing Values     : {df_processed.isnull().sum().sum()}")

assert df_processed.isnull().sum().sum() == 0, "Dataset contains missing values!"

print("\nDataset validation passed successfully.")

FINAL DATASET VALIDATION
Dataset Shape      : (7021, 51)
Duplicate Rows     : 0
Missing Values     : 0

Dataset validation passed successfully.


In [10]:
# ============================================================
# Save Processed Dataset & Preprocessing Pipeline
# ============================================================

save_dataset(df_processed, PROCESSED_DATASET_PATH, file_format="csv")
print(f"Processed dataset saved: {PROCESSED_DATASET_PATH}")

joblib.dump(preprocessor, PREPROCESSOR_PATH)
print(f"Preprocessing pipeline saved: {PREPROCESSOR_PATH}")

Processed dataset saved: D:\customer-churn-intelligence-platform\data\processed\ml_dataset_processed.csv
Preprocessing pipeline saved: D:\customer-churn-intelligence-platform\artifacts\preprocessor.pkl


In [11]:
# ============================================================
# Feature Engineering Summary
# ============================================================

print_section("FEATURE ENGINEERING COMPLETED")

print(f"Original Dataset Shape     : {df.shape}")
print(f"Processed Dataset Shape    : {df_processed.shape}")

print(f"\nOriginal Features          : {df.shape[1] - 1} (excluding target)")
print(f"Engineered Features        : {len(engineered_columns)}")
print(f"Final Features             : {df_processed.shape[1] - 1} (excluding target)")
print(f"Target Column              : {TARGET_COLUMN}")

print(f"\nArtifacts Generated")
print(f"- Processed Dataset        : {PROCESSED_DATASET_PATH}")
print(f"- Preprocessor Pipeline    : {PREPROCESSOR_PATH}")

print(f"\nNotebook Status            : COMPLETED")

FEATURE ENGINEERING COMPLETED
Original Dataset Shape     : (7021, 20)
Processed Dataset Shape    : (7021, 51)

Original Features          : 19 (excluding target)
Engineered Features        : 7
Final Features             : 50 (excluding target)
Target Column              : Churn

Artifacts Generated
- Processed Dataset        : D:\customer-churn-intelligence-platform\data\processed\ml_dataset_processed.csv
- Preprocessor Pipeline    : D:\customer-churn-intelligence-platform\artifacts\preprocessor.pkl

Notebook Status            : COMPLETED
